# Chapter 4: Real world data representation

In [1]:
import imageio.v2 as imageio
img_arr = imageio.imread('../data/isla.jpeg')
img_arr.shape

(4032, 3024, 3)

img_arr is a NumPy array-like object with three dimensions: two spatial
dimensions, width and height, and a third dimension corresponding to the red,
green, and blue channels.


#### Changing the layout
We can use the tensor’s permute method with the old dimensions for each new
dimension to get to an appropriate layout. Given an input tensor H × W × C as
obtained previously, we get a proper layout by having channel 2 first and then channels 0 and 1:

In [2]:
import torch

img = torch.from_numpy(img_arr)
out = img.permute(2, 0, 1)
out.shape

torch.Size([3, 4032, 3024])

We’ve seen this previously, but note that this operation does not make a copy of the tensor data. Instead, out uses the same underlying storage as img and only plays with the size
and stride information at the tensor level. This is convenient because the operation is
very cheap—but just as a heads-up, changing a pixel in img will lead to a change in out.

We can preallocate a tensor of appropriate size and fill it with images loaded from
a directory, like so:

In [3]:
import os
data_dir = '../data/image-cats/'
png_files = [f for f in os.listdir(data_dir) if f.endswith('.png')]
batch = torch.zeros(len(png_files), 3, 256, 256, dtype=torch.uint8)

This snippet indicates that our batch will consist of len(png_files) RGB images 256
pixels in height and 256 pixels in width. Notice the type of the tensor: we’re expecting
each color to be represented as an 8-bit integer, as in most photographic formats from
standard consumer cameras.

In [4]:
for i, filename in enumerate(png_files):
    img_arr = imageio.imread(os.path.join(data_dir, filename))
    img_t = torch.from_numpy(img_arr)
    img_t = img_t.permute(2, 0, 1)

    # only the first three channels. Sometimes images also have an alpha
    # channel indicating transparency, but our network only wants RGB input.
    img_t = img_t[:3]
    batch[i] = img_t

###  Normalizing the data

We mentioned earlier that neural networks usually work with floating-point tensors as
their input. Neural networks exhibit the best training performance when the input
data ranges roughly from 0 to 1, or from –1 to 1 (this is an effect of how their building
blocks are defined).

 A typical thing we’ll want to do is cast a tensor to floating-point and normalize the
values of the pixels. Casting to floating-point is easy, but normalization is trickier, as it
depends on what range of the input we decide should lie between 0 and 1 (or –1 and
1). One possibility is to divide the values of the pixels by 255 (the maximum representable number in 8-bit unsigned):

In [5]:
batch = batch.float()
batch /= 255.0

Another possibility is to compute the mean and standard deviation of the input data
and scale it so that the output has zero mean and unit standard deviation across each
channel—a technique commonly known as standardization:

In [6]:
n_channels = batch.shape[1]
for c in range(n_channels):
    mean = torch.mean(batch[:, c])
    std = torch.std(batch[:, c])
    batch[:, c] = (batch[:, c] - mean) / std

C:\Users\brend\AppData\Local\Temp\ipykernel_8144\1322736906.py:4: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1861.)
  std = torch.std(batch[:, c])


NOTE:
Here, we normalize just a single batch of images because we do not know
yet how to operate on an entire dataset. In working with images, it is good practice to compute the mean and standard deviation on all the training data in
advance and then subtract and divide by these fixed, precomputed quantities.

We can perform several other operations on inputs, such as geometric transformations like rotations, scaling, and cropping. These may help with training or may be
required to make an arbitrary input conform to the input requirements of a network,
like the size of the image. We will stumble on quite a few of these strategies in section
12.6. For now, just remember that you have image-manipulation options available.

Let’s load our file and turn the resulting NumPy array
into a PyTorch tensor.

In [7]:
import csv
import numpy as np

wine_path = "../data/winequality-white.csv"
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";",
skiprows=1)
wineq_numpy

array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
       [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
       [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
       ...,
       [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
       [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
       [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]],
      shape=(4898, 12), dtype=float32)

Let’s check that all the data
has been read:

In [8]:
col_list = next(csv.reader(open(wine_path), delimiter=';'))
wineq_numpy.shape, col_list

((4898, 12),
 ['fixed acidity',
  'volatile acidity',
  'citric acid',
  'residual sugar',
  'chlorides',
  'free sulfur dioxide',
  'total sulfur dioxide',
  'density',
  'pH',
  'sulphates',
  'alcohol',
  'quality'])

Now, we’ll convert the NumPy array to a PyTorch tensor:

In [9]:
wineq = torch.from_numpy(wineq_numpy)
wineq.shape, wineq.dtype

(torch.Size([4898, 12]), torch.float32)

### Representing scores
We can treat the score as a continuous variable, keep it as a real number, and perform
a regression task, or we can treat it as a label and try to guess the label from the chemical analysis in a classification task. In both approaches, we will typically remove the
score from the tensor of input data and keep it in a separate tensor, so that we can use
the score as the ground truth without it being input to our model:

In [10]:
# Selects all rows and all columns except the last
data = wineq[:, :-1]
data, data.shape

(tensor([[ 7.0000,  0.2700,  0.3600,  ...,  3.0000,  0.4500,  8.8000],
         [ 6.3000,  0.3000,  0.3400,  ...,  3.3000,  0.4900,  9.5000],
         [ 8.1000,  0.2800,  0.4000,  ...,  3.2600,  0.4400, 10.1000],
         ...,
         [ 6.5000,  0.2400,  0.1900,  ...,  2.9900,  0.4600,  9.4000],
         [ 5.5000,  0.2900,  0.3000,  ...,  3.3400,  0.3800, 12.8000],
         [ 6.0000,  0.2100,  0.3800,  ...,  3.2600,  0.3200, 11.8000]]),
 torch.Size([4898, 11]))

In [11]:
target = wineq[:, -1]
target, target.shape

(tensor([6., 6., 6.,  ..., 6., 7., 6.]), torch.Size([4898]))

If we want to transform the target tensor into a tensor of labels, we have two options,
depending on the strategy or what we use the categorical data for. One is simply to
treat labels as an integer vector of scores:

In [12]:
target = wineq[:, -1].long()
target

tensor([6, 6, 6,  ..., 6, 7, 6])

If targets were string labels, like wine color, assigning an integer number to each string
would let us follow the same approach.


#### One-hot encoding
The other approach is to build a one-hot encoding of the scores—that is, encode each
of the 10 scores in a vector of 10 elements, with all elements set to 0 but one, at a different index for each score. This way, a score of 1 could be mapped onto the vector
(1,0,0,0,0,0,0,0,0,0), a score of 5 onto (0,0,0,0,1,0,0,0,0,0), and so on. Note
that the fact that the score corresponds to the index of the nonzero element is purely
incidental: we could shuffle the assignment, and nothing would change from a classification standpoint.

In [13]:
target_onehot = torch.zeros(target.shape[0], 10)
target_onehot.scatter_(1, target.unsqueeze(1), 1.0)

random_indices = torch.randint(0, len(target), (5,))
print("Random indices: ", target[random_indices])
print("One-hot encoding: ", target_onehot[random_indices])

Random indices:  tensor([6, 6, 5, 5, 5])
One-hot encoding:  tensor([[0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]])


Let’s see what scatter_ does. First, we notice that its name ends with an underscore. That indicates the
method will not return a new tensor, but will instead modify the tensor in place. The
arguments for scatter_ are as follows:
- The dimension along which to apply the indices. For this example, dimension 0
would scatter the indices across rows, and dimension 1 would scatter them
across columns.
- A column tensor indicating the indices of the elements to scatter.
- A tensor containing the elements to scatter or a single scalar to scatter (1, in
this case).

In other words, the previous invocation reads, “For each row, take the index of the target label (which coincides with the score in our case) and use it as the column index
to set the value 1.0.” The end result is a tensor encoding categorical information.

The second argument of scatter_, the index tensor, is required to have the same
number of dimensions as the tensor we scatter into. Since target_onehot has two
dimensions (4,898 × 10), we need to add an extra dummy dimension to target using
unsqueeze:

The call to unsqueeze adds a singleton dimension, from a 1D tensor of 4,898 elements
to a 2D tensor of size (4,898 × 1), without changing its contents. No extra elements are
added; we just decided to use an extra index to access the elements. That is, we access
the first element of target as target[0] and the first element of its unsqueezed counterpart as target_unsqueezed[0,0].

PyTorch allows us to use class indices directly as targets while training neural networks. However, if we wanted to use the score as a categorical input to the network, we
would have to transform it to a one-hot-encoded tensor.

### When to categorize
You may
wonder what the deal is with the ordinal case. There is no general recipe for it; most
commonly, such data is either treated as categorical (losing the ordering part and
hoping that maybe our model will pick it up during training if we only have a few categories) or continuous (introducing an arbitrary notion of distance).

 Let’s first obtain the mean and standard deviations for
each column:

In [14]:
data_mean = torch.mean(data, dim=0)
data_mean

tensor([6.8548e+00, 2.7824e-01, 3.3419e-01, 6.3914e+00, 4.5772e-02, 3.5308e+01,
        1.3836e+02, 9.9403e-01, 3.1883e+00, 4.8985e-01, 1.0514e+01])

In [15]:
data_var = torch.var(data, dim=0)
data_var

tensor([7.1211e-01, 1.0160e-02, 1.4646e-02, 2.5726e+01, 4.7733e-04, 2.8924e+02,
        1.8061e+03, 8.9455e-06, 2.2801e-02, 1.3025e-02, 1.5144e+00])

In this case, dim=0 indicates that the reduction is performed along dimension 0.

dim=0 (Rows): This tells PyTorch to collapse the row dimension (the 4,898 rows) by averaging the values down each column.Output
Shape: The resulting data_mean tensor will have a shape of torch.Size([11]).

At this point, we can normalize the data by subtracting the mean and dividing by the
standard deviation, which helps with the learning process (we’ll discuss this topic in
more detail in chapter 5):

In [16]:
data_normalized = (data - data_mean) / torch.sqrt(data_var)
data_normalized

tensor([[ 1.7208e-01, -8.1761e-02,  2.1326e-01,  ..., -1.2468e+00,
         -3.4915e-01, -1.3930e+00],
        [-6.5743e-01,  2.1587e-01,  4.7996e-02,  ...,  7.3995e-01,
          1.3422e-03, -8.2419e-01],
        [ 1.4756e+00,  1.7450e-02,  5.4378e-01,  ...,  4.7505e-01,
         -4.3677e-01, -3.3663e-01],
        ...,
        [-4.2043e-01, -3.7940e-01, -1.1915e+00,  ..., -1.3130e+00,
         -2.6153e-01, -9.0545e-01],
        [-1.6054e+00,  1.1666e-01, -2.8253e-01,  ...,  1.0049e+00,
         -9.6251e-01,  1.8574e+00],
        [-1.0129e+00, -6.7703e-01,  3.7852e-01,  ...,  4.7505e-01,
         -1.4882e+00,  1.0448e+00]])

### Finding thresholds

Next, let’s start to look at the data with an eye to whether an easy way exists to tell
good and bad wines apart at a glance. First, we’re going to determine which rows in
target correspond to a score less than or equal to 3:


In [17]:
bad_indexes = target <= 3
bad_indexes.shape, bad_indexes.dtype, bad_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(20))

In [18]:
bad_indexes

tensor([False, False, False,  ..., False, False, False])

Note that only 20 of the bad_indexes entries are set to True! By using a feature in
PyTorch called advanced indexing, we can use a tensor with data type torch.bool to
index the data tensor. This will essentially filter data to be only items (rows) corresponding to True in the indexing tensor. The bad_indexes tensor has the same shape
as target, with values of False or True, depending on the outcome of the comparison
between our threshold and each element in the original target tensor:

In [19]:
bad_data = data[bad_indexes]
bad_data.shape

torch.Size([20, 11])

In [20]:
bad_data

tensor([[8.5000e+00, 2.6000e-01, 2.1000e-01, 1.6200e+01, 7.4000e-02, 4.1000e+01,
         1.9700e+02, 9.9800e-01, 3.0200e+00, 5.0000e-01, 9.8000e+00],
        [5.8000e+00, 2.4000e-01, 4.4000e-01, 3.5000e+00, 2.9000e-02, 5.0000e+00,
         1.0900e+02, 9.9130e-01, 3.5300e+00, 4.3000e-01, 1.1700e+01],
        [9.1000e+00, 5.9000e-01, 3.8000e-01, 1.6000e+00, 6.6000e-02, 3.4000e+01,
         1.8200e+02, 9.9680e-01, 3.2300e+00, 3.8000e-01, 8.5000e+00],
        [7.1000e+00, 3.2000e-01, 3.2000e-01, 1.1000e+01, 3.8000e-02, 1.6000e+01,
         6.6000e+01, 9.9370e-01, 3.2400e+00, 4.0000e-01, 1.1500e+01],
        [6.9000e+00, 3.9000e-01, 4.0000e-01, 4.6000e+00, 2.2000e-02, 5.0000e+00,
         1.9000e+01, 9.9150e-01, 3.3100e+00, 3.7000e-01, 1.2600e+01],
        [1.0300e+01, 1.7000e-01, 4.7000e-01, 1.4000e+00, 3.7000e-02, 5.0000e+00,
         3.3000e+01, 9.9390e-01, 2.8900e+00, 2.8000e-01, 9.6000e+00],
        [7.9000e+00, 6.4000e-01, 4.6000e-01, 1.0600e+01, 2.4400e-01, 3.3000e+01,
         2.27

The new bad_data tensor has 20 rows, the same as the number of rows with True in the
bad_indexes tensor. It retains all 11 columns. Now we can start to get information
about wines grouped into good, middling, and bad categories. Let’s take the .mean()
of each column:

In [21]:
bad_data = data[target <= 3]
mid_data = data[(target > 3) & (target < 7)]
good_data = data[target >= 7]

bad_mean = torch.mean(bad_data, dim=0)
mid_mean = torch.mean(mid_data, dim=0)
good_mean = torch.mean(good_data, dim=0)

print('{:2} {:20} {:6} {:6} {:6}'.format('No.', 'Feature', 'Bad', 'Mid', 'Good'))
for i, args in enumerate(zip(col_list, bad_mean, mid_mean, good_mean)):
    print('{:2} {:20} {:6.2f} {:6.2f} {:6.2f}'.format(i, *args))

No. Feature              Bad    Mid    Good  
 0 fixed acidity          7.60   6.89   6.73
 1 volatile acidity       0.33   0.28   0.27
 2 citric acid            0.34   0.34   0.33
 3 residual sugar         6.39   6.71   5.26
 4 chlorides              0.05   0.05   0.04
 5 free sulfur dioxide   53.33  35.42  34.55
 6 total sulfur dioxide 170.60 141.83 125.25
 7 density                0.99   0.99   0.99
 8 pH                     3.19   3.18   3.22
 9 sulphates              0.47   0.49   0.50
10 alcohol               10.34  10.26  11.42


It looks like we’re on to something here: at first glance, the bad wines seem to have
higher total sulfur dioxide, among other differences. We could use a threshold on
total sulfur dioxide as a crude criterion for discriminating good wines from bad ones.
Let’s get the indices where the total sulfur dioxide column is below the midpoint we
calculated earlier, like so:

In [22]:
total_sulfur_threshold = 141.83
total_sulfur_data = data[:,6]

predicted_indexes = torch.lt(total_sulfur_data, total_sulfur_threshold)
predicted_indexes.shape, predicted_indexes.dtype, predicted_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(2727))

Our threshold implies that just over half of all the wines are going to be high quality.
Next, we’ll need to get the indices of the actually good wines:

In [23]:
actual_indexes = target > 5
actual_indexes.shape, actual_indexes.dtype, actual_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(3258))

About 500 more wines are categorized as good than our threshold predicted, so we
already have hard evidence that it’s not perfect. Now we need to see how well our predictions line up with the actual rankings. We will perform a logical “and” between our
prediction indexes and the actual good indexes (remember that each is just an array
of zeros and ones) and use that intersection of wines-in-agreement to determine how
well we did:

In [24]:
n_matches = torch.sum(actual_indexes & predicted_indexes).item()
n_predicted = torch.sum(predicted_indexes).item()
n_actual = torch.sum(actual_indexes).item()

print("Matches (good wines): ", n_matches)
print("Percentage of good wines predicted: ", (n_matches / n_predicted)*100, "%")
print("Percentage of good wines actual: ", (n_matches / n_actual)*100, "%")

Matches (good wines):  2018
Percentage of good wines predicted:  74.000733406674 %
Percentage of good wines actual:  61.93984039287906 %


We got around 2,000 wines right! Since we predicted 2,700 wines, we have a 74%
chance that if we predict a wine to be high quality, it actually is. Unfortunately, there
are 3,200 good wines, and we only identified 61% of them. Well, we got what we
signed up for; that’s barely better than random! Of course, this is all very naive: we
know for sure that multiple variables contribute to wine quality, and the relationships
between the values of these variables and the outcome (which could be the actual
score, rather than a binarized version of it) are likely more complicated than a simple
threshold on a single value.

 Indeed, a simple neural network would overcome all of these limitations, as would
a lot of other basic machine learning methods. We’ll have the tools to tackle this problem after the next two chapters, once we learn how to build our first neural network
from scratch. Let’s move on to other data types for now

### Working with time series

We want to change the row-per-hour organization so that we have one axis that increases at a rate of one day per index increment, and another axis that represents the hour of the day (independent of the date).
The third axis will be our different columns of data (i.e., weather, temperature, etc.).

In [25]:
bikes_numpy = np.loadtxt(
"../data/timeseries.csv",
dtype=np.float32,
delimiter=",",
skiprows=1,
converters={1: lambda x: float(x[8:10])}) # Converts date strings to numbers
# corresponding to the day of the month in column 1

bikes = torch.from_numpy(bikes_numpy)
bikes

tensor([[1.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 3.0000e+00, 1.3000e+01,
         1.6000e+01],
        [2.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 8.0000e+00, 3.2000e+01,
         4.0000e+01],
        [3.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 5.0000e+00, 2.7000e+01,
         3.2000e+01],
        ...,
        [1.7377e+04, 3.1000e+01, 1.0000e+00,  ..., 7.0000e+00, 8.3000e+01,
         9.0000e+01],
        [1.7378e+04, 3.1000e+01, 1.0000e+00,  ..., 1.3000e+01, 4.8000e+01,
         6.1000e+01],
        [1.7379e+04, 3.1000e+01, 1.0000e+00,  ..., 1.2000e+01, 3.7000e+01,
         4.9000e+01]])

To obtain our daily hours dataset, we need to view the same tensor in batches of 24
hours. Let’s take a look at the shape and strides of our bikes tensor:

In [26]:
bikes.shape, bikes.stride()

(torch.Size([17520, 17]), (17, 1))

That’s 17,520 hours and 17 columns. Now let’s reshape the data to have three axes—
day, hour, and our 17 columns:

In [27]:
daily_bikes = bikes.view(-1, 24, bikes.shape[1])
daily_bikes.shape, daily_bikes.stride()

(torch.Size([730, 24, 17]), (408, 17, 1))

What happened here? First, bikes.shape[1] is 17, the number of columns in the bikes
tensor. But the real crux of this code is the very important call to view: it changes the
way the tensor looks at the same data as contained in storage.

 As you learned in the previous chapter, calling view on a tensor returns a new tensor that changes the number of dimensions and the striding information, without
changing the storage. As a result, we can rearrange our tensor at basically zero cost
because no data will be copied. Our call to view requires us to provide the new shape
for the returned tensor. We use -1 as a placeholder for “however many indexes are
left, given the other dimensions and the original number of elements.”

 Remember also from the previous chapter that storage is a contiguous, linear container for numbers (floating-point, in this case). Our bikes tensor will have each row
stored one after the other in its corresponding storage, which is confirmed by the output from the call to bikes.stride() earlier.

 For daily_bikes, the stride is telling us that advancing by 1 along the hour dimension (the second dimension) requires us to advance by 17 places in the storage (or
one set of columns). Advancing along the day dimension (the first dimension) requires
us to advance by a number of elements equal to the length of a row in the storage
times 24 (here, 408, which is 17 × 24).

 We see that the rightmost dimension is the number of columns in the original
dataset. Then, in the middle dimension, we have time, split into chunks of 24 sequential hours. In other words, we now have N sequences of L hours in a day, for C channels. To get to our desired N × C × L ordering, we need to transpose the tensor:

In [28]:
daily_bikes = daily_bikes.transpose(1, 2)
daily_bikes.shape, daily_bikes.stride()

(torch.Size([730, 17, 24]), (408, 1, 17))

Ready for training

The “weather situation” variable is ordinal. It has four levels: 1 for good weather,
and 4 for, er, really bad. We could treat this variable as categorical, with levels interpreted as labels, or as a continuous variable. If we decided to go with categorical, we would turn the variable into a one-hot-encoded vector and concatenate the columns
with the dataset.

NOTE This situation could also be a case where it is useful to step off beyond
the main path. Speculatively, we could also try to reflect like categorical, but with
order more directly by generalizing one-hot encodings to mapping the ith of
our four categories here to a vector that has ones in the positions 0 … i and
zeros beyond that. Or, similar to the embeddings we discussed in section
4.5.4, we could take partial sums of embeddings, in which case it might make
sense to make those positive. As with many things we encounter in practical
work, this could be a place where trying what works for others and then experimenting in a systematic fashion is a good idea.

To make it easier to render our data, we’re going to limit ourselves to the first
day for a moment. We initialize a zero-filled matrix with a number of rows equal to
the number of hours in the day and a number of columns equal to the number of
weather levels:

In [29]:
first_day = bikes[:24].long()
weather_onehot = torch.zeros(first_day.shape[0], 4)
first_day[:,9]

tensor([1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 2, 2, 2, 2])

Then we scatter 1s into our matrix according to the corresponding level at each row.
Remember the use of unsqueeze to add a singleton dimension as we did in the previous sections:

In [30]:
weather_onehot.scatter_(
dim=1,
index=first_day[:,9].unsqueeze(1).long() - 1,  # Decreases the values by 1 because the weather
    # situation ranges from 1 to 4, while indices are zero-based
value=1.0)

tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.]])

Our day started with weather 1 and ended with 2, so that seems right.
 Last, we concatenate our matrix to our original dataset using the cat function.
Let’s look at the first of our results:

In [31]:
torch.cat((bikes[:24], weather_onehot), 1)[:1]

tensor([[ 1.0000,  1.0000,  1.0000,  0.0000,  1.0000,  0.0000,  0.0000,  6.0000,
          0.0000,  1.0000,  0.2400,  0.2879,  0.8100,  0.0000,  3.0000, 13.0000,
         16.0000,  1.0000,  0.0000,  0.0000,  0.0000]])

Here, we prescribed our original bikes dataset and our one-hot-encoded “weather situation” matrix to be concatenated along the column dimension (that is, 1). In other
words, the columns of the two datasets are stacked together, or equivalently, the new
one-hot-encoded columns are appended to the original dataset. For cat to succeed,
the tensors must have the same size along the other dimensions—the row dimension,
in this case. Note that our new last four columns are 1, 0, 0, 0, exactly as we would
expect with a weather value of 1.

 We can do the same process with the reshaped daily_bikes tensor. Remember that
it is shaped (B, C, L), where L = 24. We first create the zero tensor with the same B and
L but with the number of additional columns as C:

In [32]:
daily_weather_onehot = torch.zeros(daily_bikes.shape[0], 4,
daily_bikes.shape[2])
daily_weather_onehot.shape

torch.Size([730, 4, 24])

Then we scatter the one-hot encoding into the tensor in the C dimension. Since this
operation is performed in place, only the content of the tensor changes:

In [33]:
daily_weather_onehot.scatter_(
1, daily_bikes[:,9,:].long().unsqueeze(1) - 1, 1.0)
daily_weather_onehot.shape

torch.Size([730, 4, 24])

And we concatenate along the C dimension:

In [34]:
daily_bikes = torch.cat((daily_bikes, daily_weather_onehot), dim=1)

We mentioned earlier that this is not the only way to treat our “weather situation” variable. Indeed, its labels have an ordinal relationship, so we can pretend they are special
values of a continuous variable. We can transform the variable so that it runs from 0.0
to 1.0:

In [37]:
daily_bikes[:, 9, :] = (daily_bikes[:, 9, :] - 1.0) / 3.0

As we mentioned in the previous section, rescaling variables to the [0.0, 1.0] interval
or the [-1.0, 1.0] interval is something we’ll want to do for all quantitative variables, like temperature (column 10 in our dataset). We’ll see why later; for now, let’s just say
that it is beneficial to the training process.

 Multiple possibilities exist for rescaling variables. We can either map their range to
[0.0, 1.0] as

In [38]:
temp = daily_bikes[:, 10, :]
temp_min = torch.min(temp)
temp_max = torch.max(temp)
daily_bikes[:, 10, :] = ((daily_bikes[:, 10, :] - temp_min)
/ (temp_max - temp_min))

or subtract the mean and divide by the standard deviation:

In [39]:
temp = daily_bikes[:, 10, :]
daily_bikes[:, 10, :] = ((daily_bikes[:, 10, :] - torch.mean(temp))
/ torch.std(temp))

In the latter case, our variable will have a zero mean and unitary standard deviation. If
our variable were drawn from a Gaussian distribution, 68% of the samples would sit in
the [–1.0, 1.0] interval.

 Great, we’ve built another nice dataset, and we’ve seen how to deal with time-series
data. For this tour d’horizon, it’s important only that we get an idea of how a time
series is laid out and how we can wrangle the data in a form that a network will digest.
 Other kinds of data look like a time series, in that there is a strict ordering. Top
two on the list? Text and audio. We’ll take a look at text next, and the final section of
this chapter has links to additional examples for audio.


### Representing text:

In [35]:
with open('../data/archive/pride_and_prejudice.txt', encoding='utf8') as f:
    text = f.read()

In [36]:
lines = text.split('\n')
line = lines[200]
line

'“Impossible, Mr. Bennet, impossible, when I am not acquainted with him'

Let’s create a tensor that can hold the total number of one-hot-encoded characters for
the whole line

In [40]:
letter_t = torch.zeros(len(line), 128) # Hardcoded to 128 characters per ASCII character
letter_t.shape

torch.Size([70, 128])

letter_t holds a one-hot-encoded character per row. Now, we need to set a 1 on each
row in the correct position so that each row represents the correct character. The
index where the 1 has to be set corresponds to the index of the character in the
encoding:

In [41]:
for i, letter in enumerate(line.lower().strip()):
    letter_index = ord(letter) if ord(letter) < 128 else 0 # The text uses directional double
    # quotes, which are not valid ASCII, so we screen them out here.
    letter_t[i][letter_index] = 1

In [42]:
letter_t

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

We have one-hot encoded our sentence into a representation that a neural network
can digest. Word-level encoding can be done the same way by establishing a vocabulary and one-hot encoding sentences—sequences of words—along the rows of our
tensor. Since a vocabulary has many words, the encoding will produce very wide
encoded vectors, which may not be practical. We will see in the next section that there
is a more efficient way to represent text at the word level, using embeddings. For now,
let’s stick with one-hot encodings and see what happens.

We’ll define clean_words, which takes text and returns it in lowercase and stripped
of punctuation.

In [43]:
def clean_words(input_str):
    punctuation = '.,;:"!?”“_-'
    word_list = input_str.lower().replace('\n',' ').split()
    word_list = [word.strip(punctuation) for word in word_list]
    return word_list
words_in_line = clean_words(line)
line, words_in_line

('“Impossible, Mr. Bennet, impossible, when I am not acquainted with him',
 ['impossible',
  'mr',
  'bennet',
  'impossible',
  'when',
  'i',
  'am',
  'not',
  'acquainted',
  'with',
  'him'])

Next, let’s build a mapping of unique words to indexes in our encoding:

In [44]:
word_list = sorted(set(clean_words(text)))
word2index_dict = {word: i for (i, word) in enumerate(word_list)}
len(word2index_dict), word2index_dict['impossible']

(7261, 3394)

word2index_dict is now a dictionary with words as keys and an integer as a value. We
will use it to efficiently find the index of a word as we one-hot encode it. Let’s now focus on our sentence: we break it up into words and one-hot encode it—that is, we
populate a tensor with one one-hot-encoded vector per word. We create an empty vector and assign the one-hot-encoded values of the word in the sentence:

In [45]:
word_t = torch.zeros(len(words_in_line), len(word2index_dict))
for i, word in enumerate(words_in_line):
    word_index = word2index_dict[word]
    word_t[i][word_index] = 1
    print('{:2} {:4} {}'.format(i, word_index, word))

print(word_t.shape)

 0 3394 impossible
 1 4305 mr
 2  813 bennet
 3 3394 impossible
 4 7078 when
 5 3315 i
 6  415 am
 7 4436 not
 8  239 acquainted
 9 7148 with
10 3215 him
torch.Size([11, 7261])


 The choice between character-level and word-level encoding leaves us to make a
tradeoff. In many languages, there are significantly fewer characters than words: representing characters has us representing just a few classes, while representing words
requires us to represent a very large number of classes and, in any practical application, deal with words that are not in the dictionary. On the other hand, words convey
much more meaning than individual characters, so a representation of words is
considerably more informative by itself. Given the stark contrast between these two
options, it is perhaps unsurprising that intermediate ways have been sought, found,
and applied with great success. For example, the byte pair encoding method starts with a
dictionary of individual letters but then iteratively adds the most frequently observed
pairs to the dictionary until it reaches a prescribed dictionary size.